# Visualização 3D do coração no MM-WHS

Este notebook carrega uma imagem CT e o rótulo correspondente do MM-WHS, recorta a região indicada pelo rótulo e mostra:

- cortes axiais centrais com `matplotlib`;
- a superfície 3D da região rotulada com `k3d`.

O layout usado pelo NIfTI é `(x, y, z)`. A legenda segue a codificação anatômica do MM-WHS: LV=500, RV=600, LA=420, RA=550, Myo=205, AO=820 e PA=850.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from nibabel.funcs import as_closest_canonical

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO_ROOT / "src"))
from utils.project.datasets.ccta import discover_ccta_dataset  # noqa: E402
from utils.project.dataframe import require_series_column  # noqa: E402
from utils.project.analysis.mmwhs_heart_analysis import MMWHS_HEART_LABEL_SPECS  # noqa: E402
from utils.project.runtime.notebook_env import resolve_existing_path  # noqa: E402
from utils.utils.nifti_io import load_raw_img_and_label  # noqa: E402
from utils.utils.roi import mask_bounding_box_slices  # noqa: E402
from utils.visualization.images.volume import visualize_label_map_3d  # noqa: E402

MMWHS_PATH = resolve_existing_path(
    "MMWHS_BASE_PATH",
    [Path("/run/media/matheus/HD/DatasetsCCTA/MM-WHS-2017-Dataset")],
    "MM-WHS 2017",
)
EXAM_ID: str | None = None
LABEL_SPECS = MMWHS_HEART_LABEL_SPECS
HEART_LABEL_VALUES = tuple(LABEL_SPECS)
MARGIN_MM = 10.0
OUTPUT_HTML: Path | None = None
SLICE_ALPHA = 0.35

In [ ]:
inventory = discover_ccta_dataset("mmwhs", MMWHS_PATH)
exam_ids = require_series_column(inventory, "exam_id").astype(str)
if EXAM_ID is None:
    selected_index = 0
else:
    matching_indices = np.flatnonzero(exam_ids.eq(EXAM_ID).to_numpy())
    if matching_indices.size == 0:
        raise FileNotFoundError(f"Exame MM-WHS não encontrado: {EXAM_ID}")
    selected_index = int(matching_indices[0])

image_path_value = inventory.iloc[selected_index].get("path")
if not isinstance(image_path_value, (str, Path)):
    raise TypeError("O inventário MM-WHS não contém um caminho de imagem válido.")
image_path = Path(image_path_value)
label_path = image_path.with_name(
    image_path.name.replace("_image.nii.gz", "_label.nii.gz")
)
if not label_path.is_file():
    raise FileNotFoundError(f"Rótulo correspondente não encontrado: {label_path}")

raw_image_nii, raw_label_nii = load_raw_img_and_label(str(image_path), str(label_path))
if raw_label_nii is None:
    raise RuntimeError("O carregamento não retornou o rótulo MM-WHS.")
image_nii = as_closest_canonical(raw_image_nii)
label_nii = as_closest_canonical(raw_label_nii)

image = np.asanyarray(image_nii.dataobj).astype(np.float32, copy=False)
label = np.asanyarray(label_nii.dataobj)
zooms = image_nii.header.get_zooms()
spacing_xyz: tuple[float, float, float] = (
    float(zooms[0]),
    float(zooms[1]),
    float(zooms[2]),
)

if image.ndim != 3 or label.ndim != 3:
    raise ValueError(
        f"Imagem e rótulo devem ser 3D; shapes: {image.shape}, {label.shape}"
    )
if image.shape != label.shape:
    raise ValueError(
        f"Imagem e rótulo têm shapes diferentes: {image.shape} != {label.shape}"
    )
if not np.allclose(image_nii.affine, label_nii.affine):
    raise ValueError("Imagem e rótulo não têm a mesma transformação espacial.")

present_labels = np.unique(label)
heart_mask = np.isin(label, HEART_LABEL_VALUES)
if not np.any(heart_mask):
    raise ValueError(
        f"Nenhum voxel encontrado para HEART_LABEL_VALUES={HEART_LABEL_VALUES}. "
        f"Labels presentes: {present_labels.tolist()}"
    )

print(f"Imagem: {image_path}")
print(f"Rótulo: {label_path}")
print(f"Shape (x, y, z): {image.shape}")
print(f"Spacing (x, y, z) mm: {spacing_xyz}")
print(f"Labels usados: {HEART_LABEL_VALUES}")
print(f"Labels presentes: {present_labels.tolist()}")

In [ ]:
crop_slices = mask_bounding_box_slices(heart_mask, spacing_xyz, MARGIN_MM)
image_crop = image[crop_slices]
label_crop = label[crop_slices]
mask_crop = heart_mask[crop_slices]
print(f"Crop shape (x, y, z): {image_crop.shape}")
print(f"Crop slices: {crop_slices}")

In [ ]:
# Visualização axial central: a imagem em tons de cinza e o rótulo em vermelho.
z_center = image_crop.shape[2] // 2
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for axis, title, data, cmap in (
    (axes[0], "CT", image_crop[:, :, z_center].T, "gray"),
    (axes[1], "Rótulo", mask_crop[:, :, z_center].T, "Reds"),
    (axes[2], "Sobreposição", image_crop[:, :, z_center].T, "gray"),
):
    axis.imshow(data, cmap=cmap, origin="lower")
    axis.set_title(title)
    axis.axis("off")
axes[2].imshow(
    mask_crop[:, :, z_center].T, cmap="Reds", alpha=SLICE_ALPHA, origin="lower"
)
plt.tight_layout()
plt.show()

In [ ]:
plot = visualize_label_map_3d(
    label_crop,
    LABEL_SPECS,
    spacing=spacing_xyz,
    opacity=0.72,
    save_html_path=OUTPUT_HTML,
    display_plot=False,
    plot_name="MM-WHS: estruturas cardíacas",
)
display(plot)

## Observações

- A função `as_closest_canonical` padroniza a orientação dos dois NIfTI antes da comparação.
- Os sete valores anatômicos definidos em `LABEL_SPECS` formam a máscara cardíaca.
- O recorte usa a caixa delimitadora do rótulo e uma margem em milímetros, convertida para voxels usando o espaçamento de cada eixo.
- A malha 3D representa a superfície dos rótulos de referência, não uma segmentação produzida pelo pipeline.